# Empirical Data Inspect

This notebook only inspects empirical datasets defined by a given experiment config.
It reads the YAML config with `NodePKExperimentConfig.from_yaml(...)`, builds the
`AICMECompartmentsDataModule`, and prints concise descriptions of the available
empirical batches.

In [1]:
from pathlib import Path

from pff import config_dir
from pff.config_classes.node_pk_config import NodePKExperimentConfig
from pff.data.datasets.aicme_datasets import AICMECompartmentsDataModule

In [2]:
# Path relative to pff.config_dir.
# This avoids notebook working-directory issues when loading configs.
CONFIG_RELATIVE_PATH = Path("experiment_configs/AISTATS/aicme-t-pk/base.yaml")
CONFIG_PATH = config_dir / CONFIG_RELATIVE_PATH

# Which cache to inspect:
# False -> heldout empirical batches (leave-one-out target)
# True  -> no-heldout empirical batches (all individuals remain in context)
NO_HELDOUT = False

# Optional explicit dataset key. Leave as None to inspect the first available dataset.
DATASET_KEY = None

# Which batch in the selected dataset to inspect.
BATCH_INDEX = 0

In [3]:
cfg = NodePKExperimentConfig.from_yaml(str(CONFIG_PATH))

print(f"Resolved config root: {config_dir}")
print(f"Loaded config: {CONFIG_PATH}")
print("Experiment name:", cfg.experiment_name)
print("Configured empirical datasets:", cfg.mix_data.test_empirical_datasets)
print("Available mix_data fields:", sorted(vars(cfg.mix_data).keys()))

Resolved config root: /home/cesarali/Pharma/pff/config_files
Loaded config: /home/cesarali/Pharma/pff/config_files/experiment_configs/AISTATS/aicme-t-pk/base.yaml
Experiment name: aistats
Configured empirical datasets: ['cesarali/lenuzza-2016', 'cesarali/Theophylline']
Available mix_data fields: ['evaluate_prediction_steps_past', 'keep_tempfile', 'log_and_max', 'log_and_z', 'log_transform', 'n_of_databatches', 'n_of_permutations', 'n_of_target_individuals', 'normalize_by_max', 'normalize_time', 'recreate_tempfile', 'sample_size_for_generative_evaluation', 'sample_size_for_generative_evaluation_end_of_training', 'sample_size_for_generative_evaluation_val', 'store_in_tempfile', 'tempfile_path', 'test_empirical_datasets', 'test_size', 'tqdm_progress', 'train_size', 'val_size', 'z_score_normalization']


In [4]:
dm = AICMECompartmentsDataModule(cfg)

# Trigger empirical cache loading through the public API.
heldout_batches = dm.get_empirical_test_batches(no_heldout=False)
no_heldout_batches = dm.get_empirical_test_batches(no_heldout=True)

print("Heldout empirical dataset keys:", list(heldout_batches.keys()))
print("No-heldout empirical dataset keys:", list(no_heldout_batches.keys()))

Heldout empirical dataset keys: ['cesarali/lenuzza-2016', 'cesarali/Theophylline']
No-heldout empirical dataset keys: ['cesarali/lenuzza-2016', 'cesarali/Theophylline']


In [5]:
print("=== Heldout empirical batches ===")
dm.describe_empirical_test_batches(
    empirical_batches=heldout_batches,
    no_heldout=False,
    batch_index=0,
    print_available=True,
)

print()
print("=== No-heldout empirical batches ===")
dm.describe_empirical_test_batches(
    empirical_batches=no_heldout_batches,
    no_heldout=True,
    batch_index=0,
    print_available=True,
)

=== Heldout empirical batches ===
Available empirical datasets (heldout): ['cesarali/lenuzza-2016', 'cesarali/Theophylline']
Dataset 'cesarali/lenuzza-2016' contains 10 empirical batch(es).
  Batch 0 studies: ['Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016']
  Batch 0 drugs: ['memantine', 'omeprazole', '5-hydroxyomeprazole', 'omeprazole sulfone', 'repaglinide', 'hydroxy repaglinide', 'rosuvastatin', 'tolbutamide', '4-hydroxytolbutamide', 'dextromethorphan', 'digoxin', 'paracetamol', '1-hydroxymidazolam', 'paracetamol glucuronide', 'dextrorphan', 'caffeine (137X)', 'midazolam', 'paraxanthine (17X)']
Dataset 'cesarali/Theophylline' contains 12 empirical batch(es).
  Batch 0 studies: ['Theophylline']
  Batch 0 drugs: ['Theophylline']
Available studies: ['Lenuzza2016', 'Th

(['Lenuzza2016', 'Theophylline'],
 ['memantine',
  'omeprazole',
  '5-hydroxyomeprazole',
  'omeprazole sulfone',
  'repaglinide',
  'hydroxy repaglinide',
  'rosuvastatin',
  'tolbutamide',
  '4-hydroxytolbutamide',
  'dextromethorphan',
  'digoxin',
  'paracetamol',
  '1-hydroxymidazolam',
  'paracetamol glucuronide',
  'dextrorphan',
  'caffeine (137X)',
  'midazolam',
  'paraxanthine (17X)',
  'Theophylline'])

In [ ]:
selected_dataset_key, batch_list = dm.select_empirical_batch_list(
    dataset_key=DATASET_KEY,
    no_heldout=NO_HELDOUT,
)

selected_batch = batch_list[BATCH_INDEX]
batch_studies, batch_drugs = dm.describe_empirical_batch(selected_batch, print_available=False)

print("Selected cache:", "no_heldout" if NO_HELDOUT else "heldout")
print("Selected dataset:", selected_dataset_key)
print("Selected batch index:", BATCH_INDEX)
print("Studies in selected batch:", batch_studies)
print("Drugs in selected batch:", batch_drugs)

In [ ]:
# Shape reference:
# target_obs        : [B, It, Tobs, 1]
# target_rem_sim    : [B, It, Trem, 1]
# context_obs       : [B, Ic, Tctx, 1]
# target_obs_mask   : [B, It, Tobs]
# target_rem_sim_mask: [B, It, Trem]
# context_obs_mask  : [B, Ic, Tctx]

print("selected_batch.is_empirical:", selected_batch.is_empirical)
print("selected_batch.study_name:", selected_batch.study_name)
print("selected_batch.substance_name:", selected_batch.substance_name)
print("selected_batch.context_subject_name:", selected_batch.context_subject_name)
print("selected_batch.target_subject_name:", selected_batch.target_subject_name)
print()
print("target_obs shape:", tuple(selected_batch.target_obs.shape))
print("target_obs_time shape:", tuple(selected_batch.target_obs_time.shape))
print("target_obs_mask shape:", tuple(selected_batch.target_obs_mask.shape))
print("target_rem_sim shape:", tuple(selected_batch.target_rem_sim.shape))
print("target_rem_sim_time shape:", tuple(selected_batch.target_rem_sim_time.shape))
print("target_rem_sim_mask shape:", tuple(selected_batch.target_rem_sim_mask.shape))
print("context_obs shape:", tuple(selected_batch.context_obs.shape))
print("context_obs_time shape:", tuple(selected_batch.context_obs_time.shape))
print("context_obs_mask shape:", tuple(selected_batch.context_obs_mask.shape))
print("mask_context_individuals shape:", tuple(selected_batch.mask_context_individuals.shape))
print("mask_target_individuals shape:", tuple(selected_batch.mask_target_individuals.shape))

In [ ]:
# Full namedtuple-style batch printout.
selected_batch